# Nocturne Atlas — Colab runner

1. Locally: right-click the `nocturne_atlas` folder in Explorer → "Compress to ZIP file". You get `nocturne_atlas.zip`.
2. Run **Cell 1** below. It will prompt you to upload the zip. Pick `nocturne_atlas.zip`.
3. Run **Cell 2** to execute `run_paper.py`. Takes ~1-2 minutes on a free Colab CPU runtime.
4. Run **Cell 3** to play the two generated nocturnes inline and preview the plate images.

In [ ]:
# Cell 1 — upload nocturne_atlas.zip and install dependencies
import io, os, shutil, zipfile
from google.colab import files

WORK = '/content/nocturne_atlas'
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK, exist_ok=True)

uploaded = files.upload()
if not uploaded:
    raise SystemExit('No file uploaded.')
name, payload = next(iter(uploaded.items()))
print(f'Got {name} ({len(payload)/1e6:.1f} MB)')

with zipfile.ZipFile(io.BytesIO(payload)) as zf:
    zf.extractall('/content/_extract')

# Find the folder that actually contains run_paper.py
src_root = None
for r, _, fnames in os.walk('/content/_extract'):
    if 'run_paper.py' in fnames:
        src_root = r
        break
if src_root is None:
    raise SystemExit('run_paper.py not found in the uploaded zip.')

# Move project files to a clean working dir
for entry in os.listdir(src_root):
    shutil.move(os.path.join(src_root, entry), os.path.join(WORK, entry))
shutil.rmtree('/content/_extract', ignore_errors=True)
print(f'Project at {WORK}')

%pip install -q numpy Pillow librosa pretty_midi soundfile matplotlib scipy

In [ ]:
# Cell 2 — run the paper-register pipeline
%cd /content/nocturne_atlas
!python run_paper.py

In [ ]:
# Cell 3 — play the new nocturnes inline and preview both plates
from IPython.display import Audio, Image, display
import os

for label in ('plate_i_borealis', 'plate_ii_australis'):
    base = f'/content/nocturne_atlas/outputs/{label}_paper'
    wav = f'{base}/{label}_chart_to_nocturne.wav'
    plate = f'{base}/{label}_plate_paper.png'
    if os.path.exists(wav):
        print(label)
        display(Audio(wav))
    if os.path.exists(plate):
        display(Image(plate, width=720))